Phase 3b — Efficiency Metrics (parameters, GFLOPs, latency)

**Low-Light Object Detection via Decoupled VLM Pseudo-Labelling**

Research Objective 3 benchmarks this pipeline against published ExDark results on four axes: mAP,
parameter count, GFLOPs and latency.

## 1. Install dependencies

Same pinning as Phase 2 so the model that gets measured is the model that was trained. `fvcore` is
installed only as a fallback: the primary FLOP counter is `torch.utils.flop_counter`, which ships with
PyTorch and needs no extra package.

In [1]:
!pip install -q "rfdetr[train,loggers]" supervision pycocotools
!pip install -q fvcore 2>/dev/null

import os, json, time, platform, statistics
import torch
from rfdetr import RFDETRSmall

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU  :", torch.cuda.get_device_name(0))
try:
    import rfdetr
    print("rfdetr:", getattr(rfdetr, "__version__", "unknown"))
except Exception:
    pass

print("\nSection 1 OK.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Configuration

`RESOLUTION` must match the square resize used in training, which the Phase 2 log records as 512. FLOP
count and latency both scale with it, so a mismatch here silently invalidates the comparison.

`WARMUP` matters more than it looks. The first few GPU passes pay for CUDA context setup, kernel
autotuning and lazy allocation, and including them inflates the median by a wide margin. `ITERS` is
large enough that the median is stable across repeat runs.

`USE_TRAINED_CKPT` selects between the trained checkpoint and the COCO-pretrained base. Both give the
same answer, because fine-tuning changes weight values and not architecture; the trained checkpoint is
preferred only so the measurement is demonstrably of the deployed model.

In [2]:
RUN_TAG      = "yoloworld_v8"
DRIVE_ROOT   = "/content/drive/MyDrive/ExDark_dataset"
P1_MANIFEST  = f"{DRIVE_ROOT}/phase1_outputs_{RUN_TAG}/phase1_manifest.json"
P2_RESULTS   = f"{DRIVE_ROOT}/phase2_results_{RUN_TAG}/phase2_results.json"
RESULTS_DIR  = f"{DRIVE_ROOT}/supplementary_efficiency_{RUN_TAG}"

RESOLUTION       = 512      # must match the training/eval square resize
WARMUP           = 20       # discarded iterations
ITERS            = 100      # timed iterations
PRED_THRESHOLD   = 0.50     # detection threshold held during timing
BATCH            = 1        # single-image latency, the convention in the literature
USE_TRAINED_CKPT = True     # False = measure the COCO-pretrained base instead

# ---------------------------------------------------------------------------
# Published ExDark results from the three papers in the literature review that
# report detection numbers on this dataset.
#
# Restricted to three kinds of entry: each paper's own proposed method, the
# closest architectural peers to RF-DETR, and the Zero-DCE rows, since Zero-DCE
# is the enhancer this pipeline uses. Everything else is discarded.
#
# WSA-YOLO is deliberately excluded. Li et al. did not train it; they quote its
# figure from Hui et al. (2024) "for reference", and attribute their own lower
# number to having used no pretrained weights. It is a secondhand result under an
# unstated protocol, so it is not a valid comparator for a from-scratch table.
# ---------------------------------------------------------------------------
SOURCES = {
    "Li2026": {
        "cite":  "Li et al. (2026), DeC-YOLO",
        "venue": "Optics and Laser Technology 196:114610",
        "split": "5,891 train / 736 test / 736 val (8:1:1, following Hui et al., 2024)",
        "setup": "608x608, RTX 3080Ti, 120 epochs",
        "pretrained": "no (stated explicitly: no pre-trained weights, for a fairer comparison)",
        "n_methods": 13, "full_range": (0.728, 0.831), "best": "DeC-YOLO",
        "range_note": "13 methods trained from scratch; the quoted WSA-YOLO figure of 0.839 is excluded",
    },
    "Fu2025": {
        "cite":  "Fu et al. (2025), DAMFCN + DarkYOLOv8",
        "venue": "Alexandria Engineering Journal 126:60-69",
        "split": "5,890 train / 736 val / 737 test (8:1:1)",
        "setup": "RTX 4090, batch 32, 300 epochs, AdamW",
        "pretrained": "not stated",
        "n_methods": 11, "full_range": (0.641, 0.698), "best": "DarkYOLOv8",
        "range_note": "",
    },
    "Liu2025": {
        "cite":  "Liu et al. (2025), PFNN",
        "venue": "Frontiers in Artificial Intelligence, 10.3389/frai.2025.1622100",
        "split": "not stated for ExDark",
        "setup": "GTX 3090Ti, batch 8, 300 epochs, Adam",
        "pretrained": "yes (YOLOv5 initialised from pretrained weights, then partly frozen)",
        "n_methods": 8, "full_range": (0.6845, 0.7171), "best": "PFNN",
        "range_note": "",
    },
}

# name, source, mAP@0.5, mAP@[.5:.95], params_M, GFLOPs, FPS, why-kept
LITERATURE = [
    ("DeC-YOLO",           "Li2026",  0.831,  0.421, 10.7,  9.6, 97.6, "the paper's own method"),
    ("DETR",               "Li2026",  0.812,  0.389, 42.2, 52.6, 21.4, "DETR-family peer"),
    ("DarkYOLOv8",         "Fu2025",  0.698,  0.430,  2.9,  6.7, None, "the paper's own method"),
    ("RT-DETR",            "Fu2025",  0.679,  0.416,  3.2, 15.4, None, "DETR-family peer"),
    ("Zero-DCE + YOLOv8n", "Fu2025",  0.649,  0.395,  3.7, 10.8, None, "Zero-DCE, the enhancer used here"),
    ("PFNN",               "Liu2025", 0.7171, 0.4507, None, None, None, "the paper's own method"),
    ("ZeroDCE + YOLOv5",   "Liu2025", 0.7051, 0.4295, None, None, None, "Zero-DCE, the enhancer used here"),
]

# Reference points for the Zero-DCE rows only. A Zero-DCE figure states nothing on
# its own; the effect is the difference from the same paper's unenhanced baseline.
# Held here rather than as table rows so the comparison stays restricted.
ZERODCE_BASELINE = {
    "Fu2025":  ("YOLOv8n", 0.671, 0.406),
    "Liu2025": ("YOLOv5",  0.6845, 0.4020),
}

# Recorded, not tabulated: the two methods both Li et al. and Fu et al. report,
# which is the evidence that absolute ExDark scores are protocol-dependent.
CROSS_SOURCE = [("IA-YOLO", 0.829, 0.667), ("YOLOv8n", 0.818, 0.671)]

# This study, from Phase 2 / Phase 2b. Reported conditions only.
OURS = [("C@0.55", 0.758, 0.420), ("H25", 0.827, 0.479), ("A", 0.893, 0.634)]

os.makedirs("/content", exist_ok=True)
print(f"resolution {RESOLUTION} | warmup {WARMUP} | timed iters {ITERS} | batch {BATCH}")
print(f"literature rows: {len(LITERATURE)} from {len(SOURCES)} papers "
      f"| our conditions: {[o[0] for o in OURS]}")
for k, v in SOURCES.items():
    n = sum(1 for r in LITERATURE if r[1] == k)
    print(f"  {k:<8} {v['cite']:<38} {n} of {v['n_methods']} methods shown "
          f"| pretrained: {v['pretrained'][:20]}")
print("\nSection 2 OK.")

resolution 512 | warmup 20 | timed iters 100 | batch 1
literature rows: 7 from 3 papers | our conditions: ['C@0.55', 'H25', 'A']
  Li2026   Li et al. (2026), DeC-YOLO             2 of 13 methods shown | pretrained: no (stated explicitl
  Fu2025   Fu et al. (2025), DAMFCN + DarkYOLOv8  3 of 11 methods shown | pretrained: not stated
  Liu2025  Liu et al. (2025), PFNN                2 of 8 methods shown | pretrained: yes (YOLOv5 initiali

Section 2 OK.


## 3. Mount Drive & locate a trained checkpoint

The search is deliberately broad. Phase 2 writes its checkpoints under a per-condition output
directory whose exact name has changed across runs, so rather than hard-coding a path this cell reads
the Phase 2 results manifest where available and falls back to a recursive glob. If nothing is found
the notebook continues against the COCO-pretrained base, which measures the same architecture.

In [3]:
from google.colab import drive
import glob

def robust_mount(mp="/content/drive", tries=3):
    for i in range(tries):
        try:
            drive.mount(mp, force_remount=True); return
        except ValueError as e:
            print(f"[mount] attempt {i+1} failed: {e}")
            os.system(f"fusermount -u {mp} 2>/dev/null"); os.system(f"rm -rf {mp}")
            time.sleep(3)
    drive.mount(mp)

robust_mount()
os.makedirs(RESULTS_DIR, exist_ok=True)

# Checkpoint discovery. The previous version fell back to the COCO-pretrained
# base without stopping when the glob returned nothing, which silently measured
# the wrong model. It now searches several patterns, retries once in case Drive
# has not finished mounting, and raises rather than substituting a base model.
CKPT, CKPT_TAG = None, None

def _find_checkpoint():
    """Return (path, tag) for a trained checkpoint, or (None, None)."""
    # 1. the Phase 2 manifest, which names the condition explicitly
    if os.path.isfile(P2_RESULTS):
        try:
            p2 = json.load(open(P2_RESULTS))
            for tag, rec in (p2.get("results") or {}).items():
                best = rec.get("best") or rec.get("best_checkpoint")
                if best and os.path.isfile(best):
                    return best, tag
        except Exception as e:
            print(f"[ckpt] could not read {os.path.basename(P2_RESULTS)}: {e}")
    # 2. the known Phase 2 / 2b checkpoint directories, cheapest first
    for pat in (f"{DRIVE_ROOT}/phase2_checkpoints_{RUN_TAG}/*/checkpoint_best_ema.pth",
                f"{DRIVE_ROOT}/phase2b_checkpoints_{RUN_TAG}/*/checkpoint_best_ema.pth",
                f"{DRIVE_ROOT}/phase2_checkpoints_{RUN_TAG}/*/checkpoint_best*.pth"):
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0], os.path.basename(os.path.dirname(hits[0]))
    # 3. last resort, a full recursive sweep of Drive
    for name in ("checkpoint_best_ema.pth", "checkpoint_best_regular.pth", "checkpoint_best.pth"):
        hits = sorted(glob.glob(f"{DRIVE_ROOT}/**/{name}", recursive=True))
        if hits:
            return hits[0], os.path.basename(os.path.dirname(hits[0]))
    return None, None

if USE_TRAINED_CKPT:
    CKPT, CKPT_TAG = _find_checkpoint()
    if CKPT is None:
        # Drive occasionally reports an empty tree immediately after mounting.
        print("[ckpt] nothing found on the first pass, waiting for Drive and retrying...")
        time.sleep(10)
        CKPT, CKPT_TAG = _find_checkpoint()

if USE_TRAINED_CKPT and CKPT is None:
    raise FileNotFoundError(
        "No trained checkpoint found, and USE_TRAINED_CKPT is True.\n"
        f"Looked in:\n"
        f"  {P2_RESULTS}  (exists: {os.path.isfile(P2_RESULTS)})\n"
        f"  {DRIVE_ROOT}/phase2_checkpoints_{RUN_TAG}/*/checkpoint_best_ema.pth\n"
        f"  {DRIVE_ROOT}/phase2b_checkpoints_{RUN_TAG}/*/checkpoint_best_ema.pth\n"
        f"  {DRIVE_ROOT}/**/checkpoint_best*.pth  (recursive)\n"
        f"Drive root exists: {os.path.isdir(DRIVE_ROOT)}\n"
        "Set USE_TRAINED_CKPT = False only if you intend to measure the untrained "
        "COCO base. Note that the base carries an 80-class head against this "
        "pipeline's 13, so its parameter count is about 0.28 M higher and is NOT "
        "the figure to report.")

if CKPT:
    print(f"[ckpt] condition : {CKPT_TAG}")
    print(f"[ckpt] path      : {CKPT}")
    print(f"[ckpt] size      : {os.path.getsize(CKPT)/1e6:.0f} MB")
else:
    print("[ckpt] USE_TRAINED_CKPT is False -> measuring the COCO-pretrained base.")
    print("[ckpt] Architecture is identical, but the 80-class head makes the parameter")
    print("[ckpt] count about 0.28 M higher than the trained 13-class model.")

print("\nSection 3 OK.")

[mount] attempt 1 failed: mount failed
Mounted at /content/drive
[ckpt] condition : A
[ckpt] path      : /content/drive/MyDrive/ExDark_dataset/phase2_checkpoints_yoloworld_v8/A/checkpoint_best_ema.pth
[ckpt] size      : 128 MB

Section 3 OK.


## 4. Load the model and locate the network

`RFDETRSmall` is a wrapper, and the `torch.nn.Module` that actually holds the parameters sits a couple
of attribute hops inside it. Hard-coding `model.model.model` would break silently if the package layout
changed, and a silent break here would produce a plausible-looking but wrong parameter count. The
resolver below walks the wrapper until it finds an `nn.Module` and raises loudly if it cannot.

In [4]:
if CKPT:
    try:
        model = RFDETRSmall.from_checkpoint(CKPT)
    except Exception:
        model = RFDETRSmall(pretrain_weights=CKPT)
    SOURCE = f"{os.path.basename(CKPT)} (condition {CKPT_TAG})"
else:
    model = RFDETRSmall()
    SOURCE = "COCO-pretrained base"

def find_nn_module(obj, depth=0):
    """Walk the wrapper to the underlying nn.Module without hard-coding a path."""
    if isinstance(obj, torch.nn.Module):
        return obj
    if depth > 4:
        return None
    for attr in ("model", "module", "net", "detector"):
        if hasattr(obj, attr):
            found = find_nn_module(getattr(obj, attr), depth + 1)
            if found is not None:
                return found
    return None

net = find_nn_module(model)
if net is None:
    raise RuntimeError("Could not locate the nn.Module inside RFDETRSmall. "
                       "Inspect vars(model) and extend find_nn_module().")

DEVICE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor()
print(f"[model] {type(net).__name__} from {SOURCE}")
print(f"[model] device: {DEVICE}")
print("\nSection 4 OK.")

/usr/local/lib/python3.12/dist-packages/rfdetr/detr.py:551: FutureWarning: The `RFDETRBase` was deprecated since v1.7.0. It will be removed in v2.0.0.
  getattr(variant_obj, "__name__", symbol): variant_obj
/usr/local/lib/python3.12/dist-packages/rfdetr/detr.py:551: FutureWarning: The `RFDETRLargeDeprecated` was deprecated since v1.7.0. It will be removed in v2.0.0.
  getattr(variant_obj, "__name__", symbol): variant_obj
/usr/local/lib/python3.12/dist-packages/rfdetr/detr.py:551: FutureWarning: The `RFDETRSegPreview` was deprecated since v1.7.0. It will be removed in v2.0.0.
  getattr(variant_obj, "__name__", symbol): variant_obj
[2026-08-19 07:56:51] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-19 07:56:51] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is

[model] LWDETR from checkpoint_best_ema.pth (condition A)
[model] device: Tesla T4

Section 4 OK.


## 5. Parameter count

Counted from the network itself rather than quoted from documentation. The per-submodule breakdown is
printed because the backbone share is the interesting part: RF-DETR's size is dominated by its DINOv2
backbone, which is also the component responsible for the inherited capability that Section 4.4 of the
report measures as the zero-shot floor.

The closing line resolves the discrepancy between the 29 M figure currently in the methodology and the
32.1 M reported in the RF-DETR paper.

In [5]:
total     = sum(p.numel() for p in net.parameters())
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)

print(f"[params] total     : {total:,}  ({total/1e6:.2f} M)")
print(f"[params] trainable : {trainable:,}  ({trainable/1e6:.2f} M)")
print(f"[params] frozen    : {(total-trainable):,}\n")

BREAKDOWN = {n: sum(p.numel() for p in m.parameters()) for n, m in net.named_children()}
for n, v in sorted(BREAKDOWN.items(), key=lambda kv: -kv[1]):
    print(f"           {n:<30} {v/1e6:>7.2f} M   ({100.0*v/total:>5.1f}%)")

print(f"\n[check] methodology states 29 M | RF-DETR paper states 32.1 M | measured "
      f"{total/1e6:.1f} M")
if abs(total/1e6 - 29.0) < abs(total/1e6 - 32.1):
    print("[check] closer to the methodology figure.")
else:
    print("[check] closer to the paper figure -> correct the methodology to the measured value.")

print("\nSection 5 OK.")

[params] total     : 31,834,124  (31.83 M)
[params] trainable : 31,834,124  (31.83 M)
[params] frozen    : 0

           backbone                         23.44 M   ( 73.6%)
           transformer                       7.25 M   ( 22.8%)
           query_feat                        1.00 M   (  3.1%)
           bbox_embed                        0.13 M   (  0.4%)
           refpoint_embed                    0.02 M   (  0.0%)
           class_embed                       0.00 M   (  0.0%)

[check] methodology states 29 M | RF-DETR paper states 32.1 M | measured 31.8 M
[check] closer to the paper figure -> correct the methodology to the measured value.

Section 5 OK.


## 6. Computational cost (GFLOPs)

`FlopCounterMode` is a dispatch mode, so it intercepts the ATen operations the forward pass actually
executes. That makes it indifferent to how RF-DETR wraps its network and to the input format its
`predict` expects, which is what makes it more reliable here than a counter that has to be handed a
correctly shaped dummy tensor. One untimed call runs first so lazy initialisation is not counted.

One convention trap is worth stating in the write-up. Ultralytics-style "GFLOPs" for YOLO models are
generally MACs-based, so YOLOv8n's published 9.2 is not the same unit as a raw FLOP count. Both figures
are printed below; quote the MACs figure when comparing against the YOLO family, and say which
convention was used.

In [6]:
from PIL import Image
import numpy as np

# A real test image if one is reachable, otherwise a synthetic dark frame. FLOPs and
# latency do not depend on image content, only on the resize the model applies.
probe_img = None
for pat in (f"{DRIVE_ROOT}/ExDark/*/*.jpg", f"{DRIVE_ROOT}/ExDark/*/*.JPG"):
    hits = glob.glob(pat)
    if hits:
        probe_img = Image.open(hits[0]).convert("RGB")
        print(f"[image] {os.path.basename(hits[0])}  {probe_img.size}")
        break
if probe_img is None:
    probe_img = Image.fromarray(np.random.randint(0, 60, (RESOLUTION, RESOLUTION, 3), dtype=np.uint8))
    print(f"[image] synthetic dark frame {probe_img.size}")

GFLOPS = GMACS = None
try:
    from torch.utils.flop_counter import FlopCounterMode
    with torch.inference_mode():
        model.predict(probe_img, threshold=PRED_THRESHOLD)          # warm lazy state
        with FlopCounterMode(display=False) as fc:
            model.predict(probe_img, threshold=PRED_THRESHOLD)
    f = fc.get_total_flops()
    GFLOPS, GMACS = f / 1e9, f / 2e9
    print(f"\n[flops] {GFLOPS:.1f} GFLOPs   ({GMACS:.1f} GMACs)   at {RESOLUTION}px, batch {BATCH}")
    print("[flops] quote GMACs against YOLOv8n (9.2) and DeC-YOLO (9.6); "
          "quote GFLOPs against DETR (52.6) and Faster R-CNN (95.4)")
except Exception as e:
    print(f"\n[flops] FlopCounterMode failed: {type(e).__name__}: {e}")
    try:
        from fvcore.nn import FlopCountAnalysis
        dummy = torch.randn(BATCH, 3, RESOLUTION, RESOLUTION,
                            device="cuda" if torch.cuda.is_available() else "cpu")
        GMACS = FlopCountAnalysis(net, dummy).total() / 1e9
        GFLOPS = GMACS * 2
        print(f"[flops] fvcore fallback: {GMACS:.1f} GMACs ({GFLOPS:.1f} GFLOPs)")
    except Exception as e2:
        print(f"[flops] fvcore fallback also failed: {type(e2).__name__}: {e2}")
        print("[flops] record as not obtained rather than quoting the paper figure.")

print("\nSection 6 OK.")

[image] 2015_04051.jpg  (507, 338)


[2026-08-19 07:57:22] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).



[flops] FlopCounterMode failed: AttributeError: 'NoneType' object has no attribute 'next_functions'


backbone.0.encoder.encoder.encoder.layer.0.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.1.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.10.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.11.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.2.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.3.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.4.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.5.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.6.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.7.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.8.attention.attention.dropout, backbone.0.encoder.encoder.encoder.layer.9.attention.attention.dropout, backbone.0.projector.stages_sampling.0.0, backbone.0.projector.stages_sampling.0.1, backbone.0.projector.stages_sampling.0.2, backbone

[flops] fvcore fallback: 25.4 GMACs (50.9 GFLOPs)

Section 6 OK.


## 7. Latency and throughput

Three details decide whether a latency number means anything. `torch.cuda.synchronize()` is called
inside the loop, because a CUDA launch returns before the kernel finishes and timing without it
measures queueing rather than computation. Warmup iterations are discarded. The median is reported
rather than the mean, since the mean is pulled by occasional scheduler stalls, and p95 is printed
alongside so the spread is visible.

Timing wraps `predict()` end to end, which includes the resize and the postprocessing. That matches how
the published FPS figures are produced, since those include non-maximum suppression for the YOLO
entries. RF-DETR warns that its default path is not optimised for inference, so where an optimisation
entry point exists both paths are measured. The two differ by a large factor, and quoting the wrong one
would misrepresent the model by more than any accuracy difference in the report.

In [7]:
def bench(fn, warmup=WARMUP, iters=ITERS):
    for _ in range(warmup):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    s = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()          # never time an async launch
        s.append((time.perf_counter() - t0) * 1000.0)
    s.sort()
    return {"median_ms": statistics.median(s), "mean_ms": statistics.fmean(s),
            "p95_ms": s[int(0.95 * len(s)) - 1], "min_ms": s[0],
            "fps": 1000.0 / statistics.median(s), "iters": iters}

call = lambda: model.predict(probe_img, threshold=PRED_THRESHOLD)

with torch.inference_mode():
    LAT_FP32 = bench(call)
print(f"[fp32] median {LAT_FP32['median_ms']:.2f} ms | p95 {LAT_FP32['p95_ms']:.2f} ms | "
      f"min {LAT_FP32['min_ms']:.2f} ms -> {LAT_FP32['fps']:.1f} FPS")

LAT_FP16 = None
opt = next((n for n in ("optimize_for_inference", "inference", "export")
            if hasattr(model, n)), None)
if opt and torch.cuda.is_available():
    try:
        getattr(model, opt)(dtype=torch.float16)
        with torch.inference_mode():
            LAT_FP16 = bench(call)
        LAT_FP16["method"] = opt
        print(f"[fp16] median {LAT_FP16['median_ms']:.2f} ms | p95 {LAT_FP16['p95_ms']:.2f} ms | "
              f"min {LAT_FP16['min_ms']:.2f} ms -> {LAT_FP16['fps']:.1f} FPS "
              f"(via model.{opt})")
        print(f"[fp16] speedup over fp32: {LAT_FP32['median_ms']/LAT_FP16['median_ms']:.1f}x")
    except Exception as e:
        print(f"[fp16] optimisation failed: {type(e).__name__}: {e}")
        print("[fp16] quote the fp32 figure and state that it is un-optimised.")
else:
    print("[fp16] no optimisation entry point on this build; fp32 is the figure to quote.")

QUOTED = LAT_FP16 or LAT_FP32
QUOTED_LABEL = "fp16, optimised" if LAT_FP16 else "fp32, un-optimised"
print(f"\n[quote] {QUOTED['median_ms']:.2f} ms ({QUOTED['fps']:.1f} FPS) on {DEVICE} "
      f"[{QUOTED_LABEL}]")
print("[quote] the device name must appear next to any latency figure in the report")

print("\nSection 7 OK.")

[fp32] median 29.57 ms | p95 31.67 ms | min 27.82 ms -> 33.8 FPS


/tmp/ipykernel_993/1259864960.py:30: FutureWarning: The `optimize_for_inference` was deprecated since v1.9.0 in favor of `rfdetr.detr.inference`. It will be removed in v1.11.0.
  getattr(model, opt)(dtype=torch.float16)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
/usr/local/lib/python3.12/dist-packages/rfdetr/models/backbone/dinov2.py:273: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert x.shape[2] % block_size == 0 and x.shape[3] % block_size == 0, (
/usr/local/lib/python3.12/dist-packages/rfdetr/models/backbone/dinov2_with_windowed_attn.py:436: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so 

[fp16] median 12.92 ms | p95 25.73 ms | min 11.74 ms -> 77.4 FPS (via model.optimize_for_inference)
[fp16] speedup over fp32: 2.3x

[quote] 12.92 ms (77.4 FPS) on Tesla T4 [fp16, optimised]
[quote] the device name must appear next to any latency figure in the report

Section 7 OK.


## 8. Against the published figures


In [16]:
import pandas as pd

def _pre(srck):
    v = SOURCES[srck]["pretrained"]
    return "no" if v.startswith("no ") else ("unstated" if v.startswith("not") else "yes")

rows = []
for name, srck, m50, m95, prm, gf, fps, why in LITERATURE:
    rows.append({"Method": name, "Source": SOURCES[srck]["cite"].split(",")[0],
                 "Pretrained": _pre(srck), "mAP@0.5": m50, "mAP@[.5:.95]": m95,
                 "Params (M)": prm, "GFLOPs": gf, "FPS": fps})
for name, m50, m95 in OURS:
    rows.append({"Method": f"This study, {name}", "Source": "this work", "Pretrained": "yes",
                 "mAP@0.5": m50, "mAP@[.5:.95]": m95,
                 "Params (M)": round(total / 1e6, 1),
                 "GFLOPs": (round(GFLOPS, 1) if GFLOPS else None),
                 "FPS": round(QUOTED["fps"], 1)})

COMPARISON = pd.DataFrame(rows)
COMPARISON_SORTED = COMPARISON.sort_values("mAP@[.5:.95]", ascending=False).reset_index(drop=True)
display(COMPARISON_SORTED)

# Derived quantities, computed quietly and written to the JSON in Section 9.
SRC_RANGE = {k: v["full_range"] for k, v in SOURCES.items()}
ZDC = []
for name, srck, m50, m95, *_ in [r for r in LITERATURE if "DCE" in r[0]]:
    bn, b50, b95 = ZERODCE_BASELINE[srck]
    ZDC.append({"source": srck, "pair": f"{name} vs {bn}",
                "delta_map50": round(m50 - b50, 4), "delta_map95": round(m95 - b95, 4)})

print("\nSection 8 OK.")

,Method,Source,Pretrained,mAP@0.5,mAP@[.5:.95],Params (M),GFLOPs,FPS
0,"This study, A",this work,yes,0.8930,0.6340,31.8,50.9,77.4
1,"This study, H25",this work,yes,0.8270,0.4790,31.8,50.9,77.4
2,PFNN,Liu et al. (2025),yes,0.7171,0.4507,NaN,NaN,NaN
3,DarkYOLOv8,Fu et al. (2025),unstated,0.6980,0.4300,2.9,6.7,NaN
4,ZeroDCE + YOLOv5,Liu et al. (2025),yes,0.7051,0.4295,NaN,NaN,NaN
5,DeC-YOLO,Li et al. (2026),no,0.8310,0.4210,10.7,9.6,97.6
6,"This study, C@0.55",this work,yes,0.7580,0.4200,31.8,50.9,77.4
7,RT-DETR,Fu et al. (2025),unstated,0.6790,0.4160,3.2,15.4,NaN
8,Zero-DCE + YOLOv8n,Fu et al. (2025),unstated,0.6490,0.3950,3.7,10.8,NaN
9,DETR,Li et al. (2026),no,0.8120,0.3890,42.2,52.6,21.4



Section 8 OK.


## 9. Persist

Written next to the other supplementary outputs on Drive, in the same shape as
`supplementary_zeroshot.json`, so the figures can be read back without re-running the notebook. The
environment block is included because latency is meaningless without it.

In [9]:
EFF = {
    "run_tag": RUN_TAG,
    "source": SOURCE,
    "environment": {
        "device": DEVICE,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "resolution": RESOLUTION,
        "batch": BATCH,
        "threshold": PRED_THRESHOLD,
        "warmup": WARMUP,
        "iters": ITERS,
    },
    "params": {
        "total": total,
        "trainable": trainable,
        "total_M": round(total / 1e6, 2),
        "by_submodule": BREAKDOWN,
        "note": "identical for A / B / C@0.55 / C@0.60 / H10 / H25 / zero-shot",
    },
    "compute": {"gflops": GFLOPS, "gmacs": GMACS,
                "note": "GFLOPs = 2 x MACs. YOLO papers usually report the MACs figure."},
    "latency": {"fp32": LAT_FP32, "fp16": LAT_FP16,
                "quoted": QUOTED, "quoted_label": QUOTED_LABEL},
    "sources": SOURCES,
    "comparison": COMPARISON.to_dict("records"),
    "source_ranges": {k: [float(a), float(b)] for k, (a, b) in SRC_RANGE.items()},
    "cross_source": CROSS_SOURCE,
    "zerodce_effect": ZDC,
}
json.dump(EFF, open(os.path.join(RESULTS_DIR, "efficiency_metrics.json"), "w"),
          indent=2, default=str)
COMPARISON_SORTED.to_csv(os.path.join(RESULTS_DIR, "efficiency_comparison.csv"), index=False)

print("Saved to", RESULTS_DIR)
for f in sorted(os.listdir(RESULTS_DIR)):
    print("  -", f)

print("\n=== Headline: paste into Table 4.35 ===")
print(f"  Params   : {total/1e6:.1f} M")
print(f"  GFLOPs   : {GFLOPS:.1f}" if GFLOPS else "  GFLOPs   : not obtained")
if GMACS:
    print(f"  (GMACs)  : {GMACS:.1f}   <- use this against the YOLO entries")
print(f"  Latency  : {QUOTED['median_ms']:.2f} ms  ({QUOTED['fps']:.1f} FPS)  [{QUOTED_LABEL}]")
print(f"  Hardware : {DEVICE}")

Saved to /content/drive/MyDrive/ExDark_dataset/supplementary_efficiency_yoloworld_v8
  - efficiency_comparison.csv
  - efficiency_metrics.json

=== Headline: paste into Table 4.35 ===
  Params   : 31.8 M
  GFLOPs   : 50.9
  (GMACs)  : 25.4   <- use this against the YOLO entries
  Latency  : 12.92 ms  (77.4 FPS)  [fp16, optimised]
  Hardware : Tesla T4
